# Partie 1 — Modèles de Machine Learning classiques

Ce notebook entraîne et évalue les **trois modèles ML classiques** :
1. **Régression Logistique** — baseline linéaire, interprétable
2. **Random Forest complet** (80 features) — ensembliste, non-linéaire
3. **Random Forest sans fuite** (37 features) — démonstration méthodologique

Toutes les évaluations sur le **même test set** issu de `01_eda_lendingclub.ipynb`. Métrique principale : **AUC-ROC**.

## 1. Imports et chargement des données préparées

In [1]:
import numpy as np
import pandas as pd
import joblib
import time
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score, confusion_matrix
import seaborn as sns

X_train = np.load('../outputs/X_train.npy')
X_test  = np.load('../outputs/X_test.npy')
y_train = np.load('../outputs/y_train.npy')
y_test  = np.load('../outputs/y_test.npy')
feature_names = pd.read_csv('../outputs/feature_names.csv', header=None)[0].tolist()

print(f'X_train: {X_train.shape} | y_train défauts: {y_train.mean():.2%}')
print(f'X_test:  {X_test.shape} | y_test défauts:  {y_test.mean():.2%}')
print(f'Features: {len(feature_names)}')

X_train: (70666, 80) | y_train défauts: 20.43%
X_test:  (17667, 80) | y_test défauts:  20.43%
Features: 80


## 2. Modèle 1 — Régression Logistique

**Paramètres clés :**
- `class_weight='balanced'` : compense le déséquilibre 20/80 (1 défaut pénalisé ~4× plus)
- `max_iter=1000` : assure la convergence
- `random_state=42` : reproductibilité

In [1]:
t0 = time.time()
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
t_lr = time.time() - t0

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

auc_lr = roc_auc_score(y_test, y_proba_lr)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr)
print(f'Régression Logistique — AUC: {auc_lr:.4f} | Acc: {acc_lr:.4f} | '
      f'F1: {f1_lr:.4f} | Temps: {t_lr:.2f}s')

Régression Logistique — AUC: 0.7311 | Acc: 0.6672 | F1: 0.4476 | Temps: 0.50s


## 3. Modèle 2 — Random Forest (complet)

**Paramètres clés :**
- `n_estimators=100` : 100 arbres (compromis perf/temps)
- `min_samples_split=10` : évite le surapprentissage
- `class_weight='balanced'` : pondération du déséquilibre
- `n_jobs=-1` : parallélisation sur tous les cœurs

In [1]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, min_samples_split=10,
                            class_weight='balanced', n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
t_rf = time.time() - t0

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_proba_rf)
acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf)
print(f'Random Forest complet — AUC: {auc_rf:.4f} | Acc: {acc_rf:.4f} | '
      f'F1: {f1_rf:.4f} | Temps: {t_rf:.2f}s')

Random Forest complet — AUC: 0.7232 | Acc: 0.7834 | F1: 0.3684 | Temps: 2.30s


## 4. Modèle 3 — Random Forest SANS FUITE (réaliste)

**Le point méthodologique central du projet.**

Trois variables du dataset LendingClub — `grade`, `sub_grade`, `int_rate` — ne sont **pas** des données brutes mais la **décision de risque déjà prise par LendingClub** sur chaque client. Les utiliser comme entrées d'un modèle « prédire avant décider » est **circulaire**.

On entraîne donc un Random Forest avec exactement la même configuration, mais **en retirant ces variables** (plus `installment` et `installment_income_ratio` qui en dérivent). Résultat : **37 features** au lieu de 80.

In [1]:
LEAK = {'int_rate', 'installment', 'installment_income_ratio'}
def is_leak(c):
    return c in LEAK or c.startswith('grade_') or c.startswith('sub_grade_')

lean_idx = [i for i, c in enumerate(feature_names) if not is_leak(c)]
lean_cols = [feature_names[i] for i in lean_idx]
print(f'Features retirées : {len(feature_names) - len(lean_idx)} / {len(feature_names)}')
print(f'Features conservées : {len(lean_idx)}')

Features retirées : 43 / 80
Features conservées : 37


In [1]:
t0 = time.time()
rf_lean = RandomForestClassifier(n_estimators=100, min_samples_split=10,
                                 class_weight='balanced', n_jobs=-1, random_state=42)
rf_lean.fit(X_train[:, lean_idx], y_train)
t_rf_lean = time.time() - t0

y_pred_rf_lean = rf_lean.predict(X_test[:, lean_idx])
y_proba_rf_lean = rf_lean.predict_proba(X_test[:, lean_idx])[:, 1]

auc_rf_lean = roc_auc_score(y_test, y_proba_rf_lean)
acc_rf_lean = accuracy_score(y_test, y_pred_rf_lean)
f1_rf_lean  = f1_score(y_test, y_pred_rf_lean)
print(f'Random Forest SANS FUITE — AUC: {auc_rf_lean:.4f} | '
      f'Acc: {acc_rf_lean:.4f} | F1: {f1_rf_lean:.4f} | Temps: {t_rf_lean:.2f}s')

Random Forest SANS FUITE — AUC: 0.6941 | Acc: 0.7869 | F1: 0.3043 | Temps: 2.60s


## 5. Tableau récapitulatif des trois modèles ML

In [1]:
results_ml = pd.DataFrame([
    {'Modèle': 'Régression Logistique', 'AUC-ROC': auc_lr,
     'Accuracy': acc_lr, 'F1-score': f1_lr, 'Temps (s)': t_lr},
    {'Modèle': 'Random Forest complet', 'AUC-ROC': auc_rf,
     'Accuracy': acc_rf, 'F1-score': f1_rf, 'Temps (s)': t_rf},
    {'Modèle': 'Random Forest sans fuite', 'AUC-ROC': auc_rf_lean,
     'Accuracy': acc_rf_lean, 'F1-score': f1_rf_lean, 'Temps (s)': t_rf_lean},
]).round(4)
results_ml

Modele,AUC-ROC,Accuracy,F1-score,Temps (s)
Regression Logistique,0.7311,0.6672,0.4476,0.5
Random Forest (complet),0.7232,0.7834,0.3684,2.3
Random Forest (sans fuite),0.6941,0.7869,0.3043,2.6


## 6. Feature importance du Random Forest complet (top 15)

In [1]:
imp = pd.DataFrame({'feature': feature_names,
                    'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=imp, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('Top 15 features — Random Forest complet')
plt.tight_layout(); plt.savefig('../outputs/figures/feature_importance_rf.png', dpi=120)
plt.show()

print('Top 5 features les plus prédictives :')
print(imp.head(5).to_string(index=False))

Top 5 features les plus prédictives :
         feature  importance
        int_rate    0.099523
             dti    0.071102
loan_income_ratio   0.067960
installment_income_ratio   0.066817
      revol_util    0.063167


## 7. Sauvegarde des modèles et probabilités

In [1]:
joblib.dump(lr, '../outputs/models/logreg_npy.pkl')
joblib.dump(rf, '../outputs/models/random_forest_npy.pkl')
joblib.dump(rf_lean, '../outputs/models/random_forest_lean_npy.pkl')
np.save('../outputs/y_proba_lr.npy', y_proba_lr)
np.save('../outputs/y_proba_rf.npy', y_proba_rf)
np.save('../outputs/y_proba_rf_lean.npy', y_proba_rf_lean)
results_ml.to_csv('../outputs/results_ml.csv', index=False)
print('Sauvegarde OK')

Sauvegarde OK


## Conclusion de la Partie 1

**Trois enseignements :**

1. **Régression Logistique** (AUC 0.731) et **Random Forest complet** (AUC 0.723) sont **quasi-équivalents**. Sur ce dataset, les relations sont déjà largement linéaires.

2. La **régression logistique gagne sur tous les critères pratiques** : 4× plus rapide, interprétable (coefficients lisibles), conforme aux exigences réglementaires.

3. **La fuite gonfle l'AUC d'environ 0.03** : passer du RF complet (0.723) au RF sans fuite (0.694) **mesure précisément** l'effet des variables `grade`/`sub_grade`/`int_rate`. Le modèle réaliste est plus honnête mais structurellement moins performant.

La **Partie 2** étudie si un Deep Learning (MLP) peut faire mieux.